# 02 · Cleaning and features

Build the **player-season** table: one row per player per season (2021/22 to 2025/26).

- **Target:** the player's last market valuation of the season (`market_value_in_eur`, and its log).
- **Performance stats:** Premier League matches only. In a season with no Premier League appearances (e.g. the player was in another league) they are left blank, not zero: that means *no data*, not *performed badly*.
- **Per-90 stats:** left blank below 450 Premier League minutes in the season.

The logic lives in `src/features.py`; this notebook runs it step by step and checks the result.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src import config, features
from src.load_data import load_players, load_valuations, load_appearances

pd.set_option('display.max_columns', 40)

players = load_players()
valuations = load_valuations(players['player_id'])
appearances = load_appearances(players['player_id'])

## 1. Target: last valuation of each season

In [ ]:
target = features.latest_valuation_per_season(valuations)
print(len(valuations), 'valuations ->', len(target), 'player-seasons')
target.head()

## 2. Premier League season stats

In [ ]:
pl_stats = features.premier_league_season_stats(appearances)
pl_stats.head()

## 3. Assemble the full table

In [ ]:
df = features.build_player_season(players, valuations, appearances)
df.shape

In [ ]:
df.head()

## 4. Checks

In [ ]:
# one row per player per season
assert not df.duplicated(['player_id', 'season']).any()
assert df['market_value_in_eur'].notna().all()
print('no duplicates, no missing target')

In [ ]:
# rows and players per season
df.groupby('season').agg(rows=('player_id', 'size'), players=('player_id', 'nunique'), no_pl_data=('no_pl_data', 'sum'))

In [ ]:
# missing values per column
df.isna().mean().sort_values(ascending=False).round(3).head(12)

In [ ]:
df[['age', 'height_in_cm', 'pl_minutes', 'goals_per90', 'assists_per90', 'market_value_in_eur', 'value_change_pct']].describe().round(2)

In [ ]:
# spot-check: a few players across seasons
df[df['name'].isin(['Erling Haaland', 'Bukayo Saka'])][
    ['name', 'season', 'club_name', 'age', 'market_value_in_eur', 'pl_minutes', 'pl_goals', 'goals_per90', 'prev_value']
]

## 5. Save

In [ ]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(config.PROCESSED_DIR / 'player_season.csv', index=False)
print('saved', len(df), 'rows')